# FinExpert v2 — real-report data (FinQA + TAT-QA) + synthetic, QLoRA on Qwen3-4B

Runs on **Kaggle** (recommended) or **Colab**; the notebook detects which.

| Kaggle setup (once) | |
|---|---|
| Phone-verify your account | required for GPUs |
| Settings → Accelerator → **GPU T4 x2** | not P100 |
| Settings → **Internet on** | for pip, the repo and the model |
| Real runs: **Save Version → Save & Run All** | runs in the background; outputs kept in the version's Output tab |

What a run does: rebuilds the FinQA/TAT-QA data from the repo and checks it byte-for-byte against `audit.json`, trains on the
committed mix file, then scores (1) the fixed 600-question validation subset with the official scorers and (2) the 30 synthetic test examples.

## 1. Config (only edit this cell)

In [ ]:
RUN_NAME   = "v2-pilot-seed3407"
REPO_URL   = "https://github.com/mani1234765/finexpert.git"
REPO_REF   = "main"                                  # exact commit is recorded
MIX_FILE   = "data/external/train_mix_v2_pilot.json" # which examples to train on
EVAL_SUBSET = "data/external/eval_subset_v1.json"    # fixed validation questions (never train on these)

# v1 synthetic data (unchanged; hash-checked)
SYNTH_SHA256 = {
    "train":      "b9fe27b8981ed74ef5525852730c4f213578334ea1aec6240b708ad2968baa63",
    "validation": "165c3bf024ef6f2b3f9140116216d570ec4f7ccdb7a0c04e3264a076cc336a84",
    "test":       "851220d99e6d33cfab95334ad3ee9729452860f8aac1a04f5feb22032d5ae33d",
}

# Model + LoRA: same as v1.1. EPOCHS is 1 for the pilot (4x more data than v1).
MODEL_NAME     = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048
SEED           = 3407
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 16, 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
EPOCHS         = 1
LEARNING_RATE  = 2e-4
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
WARMUP_RATIO   = 0.05
LR_SCHEDULER   = "linear"
WEIGHT_DECAY   = 0.01
EVAL_EVERY_STEPS = 25          # validation loss on the monitoring set (not used to pick checkpoints)

# Generation
GEN_BATCH_SIZE      = 8
MAX_NEW_TOKENS_QA   = 384      # QA answers are short; hits are flagged
MAX_NEW_TOKENS_SYNTH = 700
RUN_BASELINE        = False    # base model on the same questions (costs ~as much time again)

UNSLOTH_VERSION = None         # pin after the first run (see run_manifest.json)
COLAB_USE_DRIVE = False        # Colab only: copy the run folder to Drive

## 2. Platform, GPU, install

In [ ]:
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle T4 x2: train on one GPU (Unsloth is single-GPU)
IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB  = "google.colab" in sys.modules
PLATFORM  = "kaggle" if IN_KAGGLE else ("colab" if IN_COLAB else "local")
WORK_DIR  = "/kaggle/working" if IN_KAGGLE else ("/content" if IN_COLAB else os.getcwd())
REPO_DIR  = "/tmp/finexpert-repo" if IN_KAGGLE else os.path.join(WORK_DIR, "finexpert-repo")  # not saved as output
RUN_DIR   = os.path.join(WORK_DIR, "runs", RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

import torch
assert torch.cuda.is_available(), "No GPU. Kaggle: Settings > Accelerator > GPU T4 x2. Colab: Runtime > T4 GPU."
GPU_NAME = torch.cuda.get_device_name(0)
assert "P100" not in GPU_NAME, "P100 selected; switch the accelerator to GPU T4 x2"
print(PLATFORM, "|", GPU_NAME, "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB | run dir", RUN_DIR)

In [ ]:
import subprocess
def sh(*cmd, cwd=None):
    r = subprocess.run(list(cmd), capture_output=True, text=True, cwd=cwd)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError(f"command failed: {' '.join(cmd)}")
    return r.stdout.strip()

try:
    sh("git", "ls-remote", REPO_URL, "HEAD")
except RuntimeError:
    raise RuntimeError("No internet. Kaggle: Settings > Internet on (needs phone verification).")

sh(sys.executable, "-m", "pip", "install", "-q", "unsloth" if UNSLOTH_VERSION is None else f"unsloth=={UNSLOTH_VERSION}")
print("Installed unsloth")

## 3. Repo, evaluators, and a byte-for-byte rebuild of the external data

In [ ]:
import json, hashlib
def git(*a): return sh("git", "-C", REPO_DIR, *a)
if not os.path.exists(REPO_DIR):
    sh("git", "clone", "-q", REPO_URL, REPO_DIR)
git("fetch", "-q", "--all", "--tags")
_is_branch = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--verify", "-q", f"origin/{REPO_REF}"],
                            capture_output=True).returncode == 0
git("checkout", "-q", "--detach", f"origin/{REPO_REF}" if _is_branch else REPO_REF)
REPO_COMMIT = git("rev-parse", "HEAD")
assert git("status", "--porcelain", "--untracked-files=no") == "", "repo has local changes; delete REPO_DIR and rerun"
print("Repo commit:", REPO_COMMIT)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
for m in [m for m in list(sys.modules) if m.startswith("finexpert")]:
    del sys.modules[m]
from finexpert.evaluation.report import build_report, format_report
from finexpert.evaluation.qa.report import build_qa_report, format_qa_report
from finexpert.data.external.common import SYSTEM_PROMPT
print("Evaluators loaded")

In [ ]:
# Clone FinQA / TAT-QA at the exact commits recorded in audit.json, rebuild, and verify every file hash.
P = lambda *a: os.path.join(REPO_DIR, *a)
audit = json.load(open(P("data/external/audit.json")))
for name, folder in [("finqa", "FinQA"), ("tatqa", "TAT-QA")]:
    src = audit["sources"][name]
    dest = P("data/external/raw", folder)
    url = {"finqa": "https://github.com/czyssrs/FinQA.git", "tatqa": "https://github.com/NExTplusplus/TAT-QA.git"}[name]
    if not os.path.exists(dest):
        sh("git", "clone", "-q", url, dest)
    sh("git", "-C", dest, "checkout", "-q", src["source_commit"])
sh(sys.executable, "scripts/build_external_dataset.py", cwd=REPO_DIR)

def sha(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()
for name, src in audit["sources"].items():
    for fname, expected in src["sha256"].items():
        assert sha(P("data/external", name, fname)) == expected, f"{name}/{fname} differs from audit.json"
assert git("status", "--porcelain", "--untracked-files=no") == "", "rebuild changed tracked files"
for split, expected in SYNTH_SHA256.items():
    assert sha(P("data/sft", f"{split}.jsonl")) == expected, f"synthetic {split} hash mismatch"
print("External data rebuilt and verified against audit.json; synthetic data verified.")

## 4. Load the training mix, monitoring set and evaluation sets

In [ ]:
from collections import Counter
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

mix = json.load(open(P(MIX_FILE)))
subset = json.load(open(P(EVAL_SUBSET)))
eval_ids = {i for s in subset["sources"].values() for i in s["example_ids"]}

train_records, monitor_records, qa_eval_records = [], [], []
for source, spec in mix["sources"].items():
    assert sha(P("data/external", source, "train.jsonl")) == spec["train_sha256"], f"{source} train changed since the mix was made"
    by_id = {r["example_id"]: r for r in read_jsonl(P("data/external", source, "train.jsonl"))}
    train_records += [by_id[i] for i in spec["train_ids"]]
    val = {r["example_id"]: r for r in read_jsonl(P("data/external", source, "validation.jsonl"))}
    monitor_records += [val[i] for i in spec["monitor_ids"]]
    qa_eval_records += [r for i, r in val.items() if i in eval_ids]

synth = {s: read_jsonl(P("data/sft", f"{s}.jsonl")) for s in SYNTH_SHA256}
assert sha(P(mix["synthetic"]["path"])) == mix["synthetic"]["sha256"]
train_records += synth["train"]
monitor_records += synth["validation"]

train_ids = {r["example_id"] for r in train_records}
assert not train_ids & eval_ids, "LEAK: training examples overlap the evaluation subset"
assert not train_ids & {r["example_id"] for r in monitor_records}, "LEAK: train/monitor overlap"
assert len(qa_eval_records) == len(eval_ids), "evaluation subset incomplete"
print("train  ", len(train_records), dict(Counter(r.get("source_type", "synthetic") for r in train_records)))
print("monitor", len(monitor_records), "| QA eval", len(qa_eval_records), "| synthetic test", len(synth["test"]))

## 5. Model + LoRA

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
                                                     dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=LORA_R, target_modules=TARGET_MODULES, lora_alpha=LORA_ALPHA,
                                         lora_dropout=LORA_DROPOUT, bias="none",
                                         use_gradient_checkpointing="unsloth", random_state=SEED)
model.print_trainable_parameters()

## 6. Format (ChatML by hand), exact token counts, length filter

In [ ]:
from datasets import Dataset
def roles(r): return {m["role"]: m["content"].strip() for m in r["messages"]}
def build_prompt(r):
    d = roles(r)
    return ("<|im_start|>system\n" + d["system"] + "<|im_end|>\n<|im_start|>user\n" + d["user"]
            + "<|im_end|>\n<|im_start|>assistant\n")
def to_text(r): return build_prompt(r) + roles(r)["assistant"] + "<|im_end|>\n"

def fit(records, name):
    kept, dropped = [], []
    for r in records:
        (kept if len(tokenizer(to_text(r)).input_ids) <= MAX_SEQ_LENGTH else dropped).append(r)
    lens = [len(tokenizer(to_text(r)).input_ids) for r in kept]
    print(f"{name}: kept {len(kept)}, dropped {len(dropped)} over {MAX_SEQ_LENGTH} tokens | "
          f"mean {sum(lens)/len(lens):.0f}, max {max(lens)}, total {sum(lens):,} tokens")
    return kept, [r["example_id"] for r in dropped], sum(lens)

train_fit, TRAIN_DROPPED, TRAIN_TOKENS = fit(train_records, "train")
monitor_fit, _, _ = fit(monitor_records, "monitor")
train_ds = Dataset.from_list([{"text": to_text(r)} for r in train_fit]).shuffle(seed=SEED)
monitor_ds = Dataset.from_list([{"text": to_text(r)} for r in monitor_fit])
assert all("<think>" not in t for t in train_ds["text"])

In [ ]:
import inspect, time
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

_len_key = "max_length" if "max_length" in inspect.signature(SFTConfig.__init__).parameters else "max_seq_length"
args = SFTConfig(
    dataset_text_field="text", per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM, num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO, lr_scheduler_type=LR_SCHEDULER, weight_decay=WEIGHT_DECAY, optim="adamw_8bit",
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(), logging_steps=5,
    eval_strategy="steps", eval_steps=EVAL_EVERY_STEPS, save_strategy="no", seed=SEED,
    output_dir=os.path.join(RUN_DIR, "checkpoints"), report_to="none", packing=False, dataset_num_proc=2,
    **{_len_key: MAX_SEQ_LENGTH},
)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=monitor_ds, args=args)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
labels = tokenizer.decode([t for t in trainer.train_dataset[0]["labels"] if t != -100])
assert "<|im_start|>user" not in labels and "<think>" not in labels, "prompt not masked"
print("Loss starts at:", repr(labels[:80]))

_t0 = time.time()
trainer.train()
TRAIN_MINUTES = round((time.time() - _t0) / 60, 2)
PEAK_VRAM_GB = round(torch.cuda.max_memory_reserved() / 1e9, 2)
TOKENS_PER_SEC = round(TRAIN_TOKENS * EPOCHS / (TRAIN_MINUTES * 60))
log = trainer.state.log_history
json.dump(log, open(os.path.join(RUN_DIR, "train_log.json"), "w"), indent=2)
print(f"Train {TRAIN_MINUTES} min | {TOKENS_PER_SEC} tokens/s | peak VRAM {PEAK_VRAM_GB} GB")

In [ ]:
import matplotlib.pyplot as plt
tr = [(l["step"], l["loss"]) for l in log if "loss" in l]
ev = [(l["step"], l["eval_loss"]) for l in log if "eval_loss" in l]
plt.figure(figsize=(7, 4)); plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), "o-", label="monitor (val) loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.title(RUN_NAME); plt.legend(); plt.grid(alpha=.3)
plt.savefig(os.path.join(RUN_DIR, "loss_curve.png"), dpi=120, bbox_inches="tight"); plt.show()

## 7. Evaluate: 600 validation QA questions + 30 synthetic test examples

Prompts are sorted by length before batching (less padding, much faster). Every answer records its token count and whether it hit the limit.

In [ ]:
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"
STOP_IDS = {tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")}
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

def generate(records, max_new_tokens, label):
    order = sorted(range(len(records)), key=lambda i: len(build_prompt(records[i])))
    out = [None] * len(records); t0 = time.time()
    for b in range(0, len(order), GEN_BATCH_SIZE):
        idx = order[b:b + GEN_BATCH_SIZE]
        enc = tokenizer([build_prompt(records[i]) for i in idx], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=PAD_ID)
        for i, g in zip(idx, gen):
            new = g[enc["input_ids"].shape[1]:].tolist()
            stop = next((j for j, t in enumerate(new) if t in STOP_IDS), None)
            out[i] = {"example_id": records[i]["example_id"],
                      "prediction": tokenizer.decode(new, skip_special_tokens=True).strip(),
                      "n_new_tokens": len(new) if stop is None else stop + 1, "hit_token_limit": stop is None}
    minutes = round((time.time() - t0) / 60, 2)
    hits = [p["example_id"] for p in out if p["hit_token_limit"]]
    print(f"{label}: {len(out)} answers in {minutes} min | hit token limit: {len(hits)}")
    return out, minutes, hits

def save_jsonl(name, items):
    with open(os.path.join(RUN_DIR, name), "w", encoding="utf-8") as f:
        for it in items: f.write(json.dumps(it, ensure_ascii=False) + "\n")

TATQA_DIR = P("data/external/raw/TAT-QA")
def evaluate(tag):
    qa_preds, qa_min, qa_hits = generate(qa_eval_records, MAX_NEW_TOKENS_QA, f"{tag} QA")
    sy_preds, sy_min, sy_hits = generate(synth["test"], MAX_NEW_TOKENS_SYNTH, f"{tag} synthetic")
    save_jsonl(f"{tag}_qa_predictions.jsonl", qa_preds); save_jsonl(f"{tag}_synthetic_predictions.jsonl", sy_preds)
    qa_rep = build_qa_report(qa_eval_records, {p["example_id"]: p["prediction"] for p in qa_preds}, TATQA_DIR)
    sy_rep = build_report(synth["test"], {p["example_id"]: p["prediction"] for p in sy_preds})
    meta = {"run_name": RUN_NAME, "repo_commit": REPO_COMMIT, "model": tag, "qa_token_limit_hits": qa_hits,
            "synthetic_token_limit_hits": sy_hits, "qa_minutes": qa_min, "synthetic_minutes": sy_min}
    qa_rep["run"] = meta; sy_rep["run"] = meta
    for name, rep, fmt in [(f"{tag}_qa_report", qa_rep, format_qa_report), (f"{tag}_synthetic_report", sy_rep, format_report)]:
        json.dump(rep, open(os.path.join(RUN_DIR, name + ".json"), "w", encoding="utf-8"), indent=2, ensure_ascii=False)
        open(os.path.join(RUN_DIR, name + ".txt"), "w", encoding="utf-8").write(fmt(rep))
    print(format_qa_report(qa_rep)); print(format_report(sy_rep))
    return qa_rep, sy_rep, qa_min + sy_min

ft_qa, ft_sy, FT_EVAL_MIN = evaluate("finetuned")
if RUN_BASELINE:
    with model.disable_adapter():
        base_qa, base_sy, BASE_EVAL_MIN = evaluate("base")

## 8. Manifest + save

In [ ]:
import platform, datetime, shutil
from importlib.metadata import version, PackageNotFoundError
def v(p):
    try: return version(p)
    except PackageNotFoundError: return None
VERSIONS = {p: v(p) for p in ["unsloth", "unsloth_zoo", "torch", "transformers", "trl", "peft", "accelerate",
                              "bitsandbytes", "datasets", "pandas"]}
def headline(qa, sy):
    s = qa["summary"]
    return {"finqa_accuracy": s["finqa"]["strict"]["accuracy"], "finqa_ci95": s["finqa"]["strict"]["ci95"],
            "tatqa_em": s["tatqa"]["official"]["em"], "tatqa_f1": s["tatqa"]["official"]["f1"],
            "tatqa_f1_ci95": s["tatqa"]["official"]["f1_ci95"],
            "synthetic_numeric_recall": sy["summary"]["numerical"]["numeric_recall"],
            "synthetic_classification": sy["summary"]["classification_metrics"]["accuracy"]}
manifest = {
    "run_name": RUN_NAME, "platform": PLATFORM,
    "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "repo_url": REPO_URL, "repo_commit": REPO_COMMIT, "mix_file": MIX_FILE, "eval_subset": EVAL_SUBSET,
    "data": {"train_examples": len(train_fit), "train_dropped_over_length": TRAIN_DROPPED,
             "train_tokens": TRAIN_TOKENS, "monitor_examples": len(monitor_fit), "qa_eval": len(qa_eval_records)},
    "model_name": MODEL_NAME,
    "training": {"seed": SEED, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
                 "target_modules": TARGET_MODULES, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
                 "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "warmup_ratio": WARMUP_RATIO,
                 "lr_scheduler": LR_SCHEDULER, "weight_decay": WEIGHT_DECAY, "max_seq_length": MAX_SEQ_LENGTH,
                 "precision": "bf16" if is_bfloat16_supported() else "fp16", "loss_masking": "assistant only"},
    "timing": {"train_minutes": TRAIN_MINUTES, "tokens_per_second": TOKENS_PER_SEC, "peak_vram_gb": PEAK_VRAM_GB,
               "finetuned_eval_minutes": FT_EVAL_MIN},
    "results": {"finetuned": headline(ft_qa, ft_sy),
                "base": headline(base_qa, base_sy) if RUN_BASELINE else None},
    "environment": {"python": platform.python_version(), "cuda": torch.version.cuda, "gpu": GPU_NAME, "packages": VERSIONS},
}
json.dump(manifest, open(os.path.join(RUN_DIR, "run_manifest.json"), "w"), indent=2)
open(os.path.join(RUN_DIR, "requirements-kaggle.txt" if IN_KAGGLE else "requirements-colab.txt"), "w").write(
    "\n".join(f"{p}=={x}" for p, x in VERSIONS.items() if x) + "\n")
model.save_pretrained(os.path.join(RUN_DIR, "adapter")); tokenizer.save_pretrained(os.path.join(RUN_DIR, "adapter"))
shutil.rmtree(os.path.join(RUN_DIR, "checkpoints"), ignore_errors=True)

import zipfile
results_zip = os.path.join(WORK_DIR, RUN_NAME + "-results.zip")  # small: everything except the adapter
with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(RUN_DIR)):
        if f != "adapter":
            z.write(os.path.join(RUN_DIR, f), f)
print(json.dumps(manifest["results"], indent=2)); print(json.dumps(manifest["timing"], indent=2))
print("Results zip (no adapter):", results_zip)

if IN_COLAB and COLAB_USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copytree(RUN_DIR, f"/content/drive/MyDrive/finexpert/runs/{RUN_NAME}", dirs_exist_ok=True)
elif IN_KAGGLE:
    print("Kaggle: everything in /kaggle/working is kept as this version's Output. Download the results zip from there.")

## Reading the results
- **QA accuracy** is on the fixed validation subset (never trained on). Quote the 95% CI; compare runs only on the same subset.
- **Strict vs lenient:** a large gap means the model reasons but doesn't follow the `Answer:` format.
- **Token-limit hits** must be ~0 for the fine-tuned model; otherwise raise `MAX_NEW_TOKENS_QA`.
- **tokens/s and minutes** in `run_manifest.json` size the full v2 runs.
- The official **test** sets are not touched here; they are scored once, for the final models.